# Competition Simulation 1: Baseline Model + CV Discipline

This notebook simulates a tabular Kaggle regression competition and builds an **honest baseline** before any fancy modeling.

**Goals**
1. Build a reproducible synthetic train/test dataset with temporal drift.
2. Compare naive random KFold vs time-aware CV and keep the competition-safe strategy.
3. Produce baseline OOF diagnostics so future iterations have trustworthy feedback.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## 1) Build a competition-like dataset (no manual downloads)

We create synthetic data with:
- informative numeric features,
- seasonality,
- a gradual trend,
- and a regime shift late in time.

This lets us demonstrate why random CV is optimistic when the leaderboard period is temporally different.


In [ ]:
n_total = 3600
n_train = 2800
time_idx = np.arange(n_total)

x1 = rng.normal(0, 1, n_total)
x2 = rng.normal(0, 1, n_total)
x3 = rng.normal(0, 1, n_total)
x4 = rng.normal(0, 1, n_total)
x5 = rng.normal(0, 1, n_total)

seasonal = np.sin(time_idx / 28) + 0.4 * np.cos(time_idx / 9)
regime = (time_idx > int(0.62 * n_total)).astype(float)
noise = rng.normal(0, 1.0, n_total)

target = (
    2.4 * x1
    - 1.8 * x2
    + 1.3 * x1 * x3
    + 0.7 * x4**2
    + 1.1 * seasonal
    + 0.0035 * time_idx
    + 0.9 * regime * x2
    + noise
)

data = pd.DataFrame(
    {
        "id": np.arange(n_total),
        "time_idx": time_idx,
        "x1": x1,
        "x2": x2,
        "x3": x3,
        "x4": x4,
        "x5": x5,
        "seasonal": seasonal,
        "target": target,
    }
)

train_df = data.iloc[:n_train].copy()
test_df = data.iloc[n_train:].copy()

feature_cols = ["time_idx", "x1", "x2", "x3", "x4", "x5", "seasonal"]
X_train = train_df[feature_cols]
y_train = train_df["target"].to_numpy()
X_test = test_df[feature_cols]

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
train_df.head()

## 2) Choose the correct CV strategy

Because the simulated leaderboard data comes later in time, **TimeSeriesSplit** is the right baseline CV.
We still compute random KFold to show the common optimistic mistake.


In [ ]:
baseline_model = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("ridge", Ridge(alpha=2.0)),
    ]
)

def evaluate_splitter(splitter, X, y):
    rows = []
    for fold, (tr_idx, va_idx) in enumerate(splitter.split(X), start=1):
        model = baseline_model
        model.fit(X.iloc[tr_idx], y[tr_idx])
        pred = model.predict(X.iloc[va_idx])
        rows.append(
            {
                "fold": fold,
                "rmse": np.sqrt(mean_squared_error(y[va_idx], pred)),
            }
        )
    return pd.DataFrame(rows)

kfold_scores = evaluate_splitter(
    KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED), X_train, y_train
)
tscv_scores = evaluate_splitter(TimeSeriesSplit(n_splits=5), X_train, y_train)

comparison = pd.DataFrame(
    {
        "strategy": [
            "Random KFold (optimistic for temporal drift)",
            "TimeSeriesSplit (competition-safe)",
        ],
        "mean_rmse": [kfold_scores["rmse"].mean(), tscv_scores["rmse"].mean()],
        "std_rmse": [kfold_scores["rmse"].std(ddof=0), tscv_scores["rmse"].std(ddof=0)],
    }
)
comparison

## 3) Baseline training with OOF diagnostics

We keep the time-aware CV and collect OOF predictions/fold metrics.
This gives a realistic offline proxy for leaderboard behavior and highlights model failure modes.


In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
oof_pred = np.full(len(X_train), np.nan)
fold_rows = []

for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_train), start=1):
    model = Pipeline([("scale", StandardScaler()), ("ridge", Ridge(alpha=2.0))])
    model.fit(X_train.iloc[tr_idx], y_train[tr_idx])
    fold_pred = model.predict(X_train.iloc[va_idx])
    oof_pred[va_idx] = fold_pred

    fold_rows.append(
        {
            "fold": fold,
            "train_rows": len(tr_idx),
            "valid_rows": len(va_idx),
            "train_time_max": int(train_df.iloc[tr_idx]["time_idx"].max()),
            "valid_time_min": int(train_df.iloc[va_idx]["time_idx"].min()),
            "valid_time_max": int(train_df.iloc[va_idx]["time_idx"].max()),
            "rmse": np.sqrt(mean_squared_error(y_train[va_idx], fold_pred)),
            "mae": mean_absolute_error(y_train[va_idx], fold_pred),
        }
    )

fold_metrics = pd.DataFrame(fold_rows)
valid_mask = ~np.isnan(oof_pred)

oof_rmse = np.sqrt(mean_squared_error(y_train[valid_mask], oof_pred[valid_mask]))
oof_mae = mean_absolute_error(y_train[valid_mask], oof_pred[valid_mask])

print(f"OOF RMSE (TimeSeriesSplit): {oof_rmse:.4f}")
print(f"OOF MAE  (TimeSeriesSplit): {oof_mae:.4f}")
fold_metrics

## 4) Diagnostics

Three quick checks:
1. Actual vs prediction calibration,
2. residual drift across time,
3. residual distribution shape.


In [ ]:
valid_mask = ~np.isnan(oof_pred)
actual = y_train[valid_mask]
predicted = oof_pred[valid_mask]
residuals = actual - predicted
valid_time = train_df.loc[valid_mask, "time_idx"]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

axes[0].scatter(actual, predicted, alpha=0.25, s=12)
min_v, max_v = actual.min(), actual.max()
axes[0].plot([min_v, max_v], [min_v, max_v], "r--", lw=1.5)
axes[0].set_title("Actual vs OOF prediction")
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("OOF prediction")

axes[1].scatter(valid_time, residuals, alpha=0.25, s=12, color="tab:orange")
axes[1].axhline(0, color="black", lw=1)
axes[1].set_title("Residuals across time")
axes[1].set_xlabel("time_idx")
axes[1].set_ylabel("Residual (actual - pred)")

axes[2].hist(residuals, bins=35, color="tab:green", alpha=0.8)
axes[2].set_title("Residual distribution")
axes[2].set_xlabel("Residual")
axes[2].set_ylabel("Count")

plt.tight_layout()

## 5) Fit final baseline model for inference

We now train on full training data and generate test predictions (submission skeleton).


In [ ]:
final_model = Pipeline([("scale", StandardScaler()), ("ridge", Ridge(alpha=2.0))])
final_model.fit(X_train, y_train)
test_predictions = final_model.predict(X_test)

submission_preview = pd.DataFrame(
    {"id": test_df["id"].to_numpy(), "prediction": test_predictions}
)
submission_preview.head()

### Baseline takeaway

The baseline is intentionally simple, but it is built with the right CV discipline and diagnostics.
That makes later feature/model improvements measurable and trustworthy.
